In [7]:
# ===============================
# 1. IMPORTS
# ===============================
import os
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score


# ===============================
# 2. BASE DIRECTORY + FILE PATH
# ===============================
import os

BASE_DIR = os.path.dirname(os.getcwd())

file_path = os.path.join(
    BASE_DIR,
    "Data",
    "raw",
    "semeval",
    "SemEval2018-T3-train-taskA_emoji_ironyHashtags.txt"
)

print("File Exists:", os.path.exists(file_path))
# ===============================
# 3. LOAD DATA
# ===============================
df = pd.read_csv(
    file_path,
    sep="\t",
    skiprows=1,   # ✅ skip header row
    header=None,
    names=["index", "label", "text"]
)

# ===============================
# 4. CLEANING FUNCTION
# ===============================
def clean_text(text):
    text = str(text)

    # remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # remove mentions (@user)
    text = re.sub(r"@\w+", "", text)

    # remove '#' but keep word
    text = re.sub(r"#(\w+)", r"\1", text)

    # normalize spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


df["text"] = df["text"].apply(clean_text)


# ===============================
# 5. CLASS DISTRIBUTION
# ===============================
print("\nClass Distribution:")
print(df["label"].value_counts())


# ===============================
# 6. TRAIN-TEST SPLIT
# ===============================
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

print("\nTrain size:", len(train_df))
print("Test size:", len(test_df))


# ===============================
# 7. TF-IDF VECTORIZATION
# ===============================
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

X_train = vectorizer.fit_transform(train_df["text"])
X_test = vectorizer.transform(test_df["text"])

y_train = train_df["label"]
y_test = test_df["label"]


# ===============================
# 8. TRAIN MODEL (BASELINE)
# ===============================
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)


# ===============================
# 9. EVALUATION
# ===============================
y_pred = model.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

# ===============================
# 10. SAVE PROCESSED FILES
# ===============================

import os

# Go to project root
BASE_DIR = os.path.dirname(os.getcwd())

# Create processed folder inside Data
output_dir = os.path.join(BASE_DIR, "Data", "processed")
os.makedirs(output_dir, exist_ok=True)

# Save files
df.to_csv(os.path.join(output_dir, "semeval_clean.csv"), index=False)
train_df.to_csv(os.path.join(output_dir, "train.csv"), index=False)
test_df.to_csv(os.path.join(output_dir, "test.csv"), index=False)

print("Files saved in:", output_dir)

File Exists: True

Class Distribution:
label
0    1916
1    1901
Name: count, dtype: int64

Train size: 3053
Test size: 764

Accuracy: 0.8363874345549738

Classification Report:

              precision    recall  f1-score   support

           0       0.86      0.80      0.83       384
           1       0.81      0.87      0.84       380

    accuracy                           0.84       764
   macro avg       0.84      0.84      0.84       764
weighted avg       0.84      0.84      0.84       764

Files saved in: C:\Users\dell\Sarcasm_detection_socialmedia_NLP\Data\processed
